# tpattern — the worked example, step by step

This notebook is the companion to the *Behavior Research Methods* tutorial: it runs the **worked example (Section 4)** end to end, in the browser, with nothing to install locally. Run the cells in order.

T-pattern analysis applies to **any** stream of timed, categorised events — behavioural coding, animal behaviour, clinical or physiological logs, human–computer interaction, conversation. Here the worked instance is the 2022 World Cup goal sequences; to analyse your own data, replace the input table in Step 1 and re-run.

**What you will do:** load events → inspect and diagnose → let the tool recommend the settings → calibrate against a chance model → read the result and its auto-generated Methods paragraph.

## Step 0 — Install and first run

Install the package and get the shipped data and examples, then run a first analysis on the small **synthetic** dataset so you reach a result immediately.

In [ ]:
#@title Install tpattern + fetch data/examples { display-mode: "form" }
!git clone -q https://github.com/ajcallaway/TPattern.git
%pip install -q -e TPattern[gui]
print('\nStep 0 — first run on the shipped synthetic ground-truth data:')
!python TPattern/examples/quickstart.py

## Step 1 — Load

Read the events from a single flat table (one row per event). Naming the observation-window columns means each sequence's window length *T* is taken from the data, not inferred from the first and last event.

In [ ]:
from tpattern import read_table

observations = read_table("TPattern/data/worldcup_goals.csv",
                          obs_start="obs_start", obs_end="obs_end", time_unit="ms")
print(f"{len(observations)} sequences, "
      f"{sum(len(o.events) for o in observations)} events")

## Step 2 — Inspect

The tool tests whether the analytic assumptions hold (conditional uniformity, dispersion). Where they fail — as they do here — it flags that the analytic *p* must not be thresholded directly and that surrogate calibration is required.

In [ ]:
from tpattern import recommend

rec = recommend(observations)
print(rec)   # dataset summary + the three recommended settings, with the reasons

## Step 3 — Recommendation and Methods text

From the same measurements the advisor recommends the surrogate null, the minimum lag, and the error-rate control — and writes the Methods sentence that justifies each, straight from the data.

In [ ]:
print(rec.methods_text())

## Step 4 — Run and read

Detect and calibrate under the recommended settings (profile-preserving null, minimum lag of one frame, *B* = 2,000 surrogates — a few minutes). Every detected pattern is tested against the chance model with false-discovery correction; the survivors are the reportable result.

In [ ]:
from tpattern import Config, calibrate, report

result = calibrate(observations, Config(min_lag=1),
                   null="profile", B=2000, q_target=0.05)

survivors = sorted(result.kept("fdr"), key=lambda c: c.fdr_q)
print(f"{len([c for c in result.real if c.level >= 1])} composite patterns detected; "
      f"{len(survivors)} survive false-discovery control:\n")
for c in survivors:
    print(f"  {c.pattern.signature():55s}  N={c.N:2d}  p={c.p_emp:.4f}  q={c.fdr_q:.3f}")

report(result, "goals_out", title="Goals")

In [ ]:
#@title Show the pattern dendrogram and the generated report { display-mode: "form" }
from IPython.display import Image, display
import os
for f in ("patterns_overview.png",):
    p = os.path.join("goals_out", f)
    if os.path.exists(p):
        display(Image(p))
print(open("goals_out/SUMMARY.txt").read())

## Now run it on your own data

Replace the file in **Step 1** with your own flat event table — one row per event, with an `observation` id, an `event` label, and a `start` time (add `obs_start`/`obs_end` if you have explicit windows) — and re-run every cell. The diagnostics, the recommended settings, the calibration and the Methods text all regenerate from whatever data you supply.

See [`SCHEMA.md`](https://github.com/ajcallaway/TPattern/blob/main/SCHEMA.md) for the (simple) input format.